# FFM_3_4 — mesh2z 收尾：只看耳廓 · 耳廓对齐 · 形状邻居是不是 z 邻居

**这是 mesh2z 的最后一个实验，规则事先写定（2026-10-04）：**
> 只截耳廓 + 耳廓对齐的最好配置，和 FFM_3_2_1 的最好 ridge（simple + ear multi-σ）在**同样的折**上
> 逐人配对比较。若端到端 dB 的改善的 95% bootstrap 区间**包含 0 或在 0 的劣侧**，mesh2z 正式结束。
> 无论结果如何，跑完即收尾。

**A. 只看耳廓、按耳廓对齐**
- 只保留比耳道平面向外伸出 > 3 mm 的点（伸出来的耳廓），去掉 45 mm 球里的脸颊/头侧皮肤
- 每只耳廓用 ICP 刚性对齐（旋转 + 平移，**不缩放**，大小是信号）到一个模板耳廓；模板由开发集自己迭代得到
- 特征：对齐后的 RFF 均值嵌入；**点对点对应**后的坐标（模板上每个点在这只耳上的最近点）；9 个笨几何量里属于耳朵的 7 个

**B. 形状上的邻居，是不是 z 上的邻居？**（不做任何回归）
- 两两距离表的秩相关（Mantel / RSA），按人置换
- 每个人形状上最近的 k 个人里，有几个也是 z 上最近的 k 个人（对比随机水平）
- 形状第一主成分 vs z 第一主成分的散点图（「一维排序」的直观版本）

**C. 回归**：同 FFM_3_2_1 的折、按人分组、每只耳一行；核 ridge；最大值置换检验；端到端 dB / LSD；逐人配对。

模板和特征标准化只用特征（不碰目标），和 FFM_3_2_1 的做法一致。36 个锁定的人不读取。

In [ ]:
# ==========================================================================
# Cell 0 — 环境 · 路径
# ==========================================================================
import os, sys, json, csv, zipfile, hashlib, time, math, warnings
warnings.filterwarnings('ignore')
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
    SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
    os.system('pip install healpy -q')
    PATHS = dict(
        latents = f'{SAVE_DIR}/mesh2z_sonicom_latents_v1.npz',
        oof     = f'{SAVE_DIR}/mesh2z_rff_v1/oof_ridge.npz',
        samples = f'{DRIVE}/ELEC5020/SONICOM/mesh2z_samples',
        ckpt    = f'{SAVE_DIR}/FFM26_best.pth',
        sonicom = f'{DRIVE}/ELEC5020/SONICOM.zip',
    )
    OUT_DIR = f'{SAVE_DIR}/mesh2z_pinna_v1'
else:
    H = os.path.expanduser('~/Documents')
    R = f'{H}/26S2/ELEC5021/hrtf-GitHub'
    PATHS = dict(
        latents = f'{R}/mesh2z/output/mesh2z_sonicom_latents_v1.npz',
        oof     = os.environ.get('MESH2Z_OOF', f'{R}/mesh2z/output/mesh2z_rff_v1/oof_ridge.npz'),
        samples = f'{H}/26S1/ELEC5020/SONICOM/mesh2z_samples',
        ckpt    = os.environ.get('FFM26_CKPT', f'{R}/results/FFM26_best.pth'),
        sonicom = f'{H}/26S1/ELEC5020/SONICOM',
    )
    OUT_DIR = os.environ.get('MESH2Z_OUT', f'{R}/mesh2z/output/mesh2z_pinna_v1')
os.makedirs(OUT_DIR, exist_ok=True)
for k, p in PATHS.items():
    print(f'{k:8s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), '先放好缺失的文件'

In [ ]:
# ==========================================================================
# Cell 1 — 配置（★ 跑之前定死）
# ==========================================================================
CFG = dict(
    PINNA_DY    = 3.0,       # mm；比耳道平面向外伸出超过这个距离的点 = 耳廓
    N_PINNA     = 4096,      # 每只耳廓用的点数（均匀子样本）
    ICP_ITERS   = 40,
    TEMPLATE_ROUNDS = 3,     # 模板迭代次数：中位耳 -> 对齐 -> 平均对应点 -> 再对齐
    N_CORR      = 1024,      # 模板上用于点对点对应的点数
    SIGMAS      = [0.005, 0.01, 0.02, 0.05, 0.1],
    M           = 256,
    ALPHA_REL   = list(np.logspace(-4, 5, 37)),
    N_INNER     = 5,
    N_PERM      = 200,
    KNN_K       = [5, 10, 20],
    N_MANTEL    = 1000,
    N_BOOT      = 2000,
    SEED        = 0,
    DID         = 2,
    NOTCH_HZ    = (6000, 12000),
)
print(json.dumps({k: (v if not isinstance(v, list) or len(v) < 8 else f'{len(v)} values') for k, v in CFG.items()}, indent=1))

In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（与 FFM_2_3 / FFM_2_6 逐字相同，构建时复制进来）
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 受试者、折、目标（对齐到 FFM_3_2_1）
# ==========================================================================
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
L = np.load(PATHS['latents']); O = np.load(PATHS['oof'])
assert str(O['latents_subj_hash']) == str(L['subj_hash'])
IDX = {s: i for i, s in enumerate(L['sid'])}
DEV = [str(s) for s in O['dev']]; NS = len(DEV)
LOCKED = {str(s) for s in L['locked_sids']}
assert set(DEV) <= {str(s) for s in L['eval_sids']} and not (set(DEV) & LOCKED)
FOLD = O['fold_of_subject']; FOLDS = sorted(set(FOLD.tolist()))
SUBJ_OF_ROW = np.repeat(np.arange(NS), 2); SIDE_OF_ROW = np.tile([0, 1], NS)
Y = np.stack([L['Z_ear'][IDX[s], side].reshape(-1) for s in DEV for side in (0, 1)]).astype(np.float64)
N_BINS = L['Z_ear'].shape[2]
RIDGE_BEST = O['pred_best'].astype(np.float64); RIDGE_NAME = f"ridge: {str(O['best'])} (FFM_3_2_1)"

def cv_r2_pred(pred):
    sse = sst = 0.0
    for f in FOLDS:
        va = np.flatnonzero(FOLD[SUBJ_OF_ROW] == f); tr = np.flatnonzero(FOLD[SUBJ_OF_ROW] != f)
        sse += ((Y[va] - pred[va]) ** 2).sum(); sst += ((Y[va] - Y[tr].mean(0)) ** 2).sum()
    return 1 - sse / sst
rj = os.path.join(os.path.dirname(PATHS['oof']), 'results.json')
if os.path.exists(rj):
    ref = json.load(open(rj))['cv_r2'][str(O['best'])]
    assert abs(cv_r2_pred(RIDGE_BEST) - ref) < 1e-4, '行或折没对齐'
    print(f'对齐检查：ridge CV-R² 重算 {cv_r2_pred(RIDGE_BEST):+.4f} = FFM_3_2_1 {ref:+.4f} ✓')
print(f'{NS} 人 / {2*NS} 只耳 | {len(FOLDS)} 折 | {device}')

In [ ]:
# ==========================================================================
# Cell 4 — 只截耳廓
# ==========================================================================
with open(os.path.join(PATHS['samples'], 'manifest.csv'), newline='') as f:
    MAN = {f"SON_{r['subject']}": r for r in csv.DictReader(f)}
assert hashlib.md5(','.join(sorted(MAN)).encode()).hexdigest()[:12] == str(L['mesh_hash'])
FLIP = np.array([1.0, -1.0, 1.0])
PINNA, CANAL, AREA, NPTS = [], [], [], []
for s in DEV:
    d = np.load(os.path.join(PATHS['samples'], f'{s[4:]}.npz'))
    for pts, canal, area in ((d['earL_pts'], d['ear_left_mm'].astype(float), float(MAN[s]['crop_area_L_mm2'])),
                             (d['earR_pts'], d['ear_right_mm'].astype(float) * FLIP, float(MAN[s]['crop_area_R_mm2']))):
        q = pts.astype(np.float64) - canal
        p = q[q[:, 1] > CFG['PINNA_DY']]                 # 点已随机打乱，过滤后仍是均匀样本
        NPTS.append(len(p)); PINNA.append(p[:CFG['N_PINNA']]); CANAL.append(canal); AREA.append(area)
NPTS = np.array(NPTS)
print(f"伸出耳廓的点数（每只耳，共 2 万点中）：中位 {np.median(NPTS):.0f}，最少 {NPTS.min()}")
assert NPTS.min() >= CFG['N_PINNA'] // 2, '有耳朵的伸出部分太少，检查 PINNA_DY'
print(f"使用每只耳 {CFG['N_PINNA']} 个点（不足的用全部）；占 45 mm 截取区的 {np.median(NPTS)/20000:.0%}")

In [ ]:
# ==========================================================================
# Cell 5 — 耳廓 ICP 对齐（刚性，不缩放），模板由开发集迭代
# ==========================================================================
from scipy.spatial import cKDTree

def kabsch(A, B):
    """找旋转 R、平移 t 使 A@R.T + t ≈ B（A、B 已逐点配对）"""
    ca, cb = A.mean(0), B.mean(0)
    U, _, Vt = np.linalg.svd((A - ca).T @ (B - cb))
    D = np.diag([1, 1, np.sign(np.linalg.det(Vt.T @ U.T))])
    R = Vt.T @ D @ U.T
    return R, cb - ca @ R.T

def icp(P, T, iters):
    tree = cKDTree(T); Rtot, ttot = np.eye(3), np.zeros(3); Q = P.copy()
    for _ in range(iters):
        _, nn = tree.query(Q)
        R, t = kabsch(Q, T[nn]); Q = Q @ R.T + t
        Rtot, ttot = R @ Rtot, R @ ttot + t
    dd, _ = tree.query(Q); rms = float(np.sqrt((dd ** 2).mean()))
    return Q, Rtot, ttot, rms

rng = np.random.default_rng(CFG['SEED'])
# 第 0 轮模板：伸出耳廓「最典型」的那只耳（简单量的中位附近）
size = np.array([np.ptp(p[:, 2]) for p in PINNA])
T = PINNA[int(np.argmin(np.abs(size - np.median(size))))][:CFG['N_CORR']].copy()
for rnd in range(CFG['TEMPLATE_ROUNDS']):
    ALIGNED, ANG, SHIFT, RMS = [], [], [], []
    for p in PINNA:
        Q, Rm, t, rms = icp(p, T, CFG['ICP_ITERS'])
        ALIGNED.append(Q); RMS.append(rms); SHIFT.append(np.linalg.norm(t))
        ANG.append(np.degrees(np.arccos(np.clip((np.trace(Rm) - 1) / 2, -1, 1))))
    # 新模板 = 模板每个点在各只对齐后耳廓上的最近点的平均
    corr = np.stack([Q[cKDTree(Q).query(T)[1]] for Q in ALIGNED])
    T = corr.mean(0)
    print(f'第 {rnd+1} 轮：残差 RMS 中位 {np.median(RMS):.2f} mm | 旋转中位 {np.median(ANG):.1f}° (90% 分位 {np.percentile(ANG,90):.1f}°)'
          f' | 平移中位 {np.median(SHIFT):.1f} mm')
CORR = corr                                              # (2NS, N_CORR, 3) 点对点对应后的坐标
print('★ 旋转若普遍是几十度，说明对齐在做「换方向」而不是微调，要检查模板')

In [ ]:
# ==========================================================================
# Cell 6 — 特征 → Gram
# ==========================================================================
G0 = torch.as_tensor(np.random.default_rng(12345).standard_normal((CFG['M'], 3)).T, dtype=torch.float32, device=device)
@torch.no_grad()
def rff(P):
    base = 2 * math.pi * (torch.as_tensor(P, dtype=torch.float32, device=device) @ G0)
    return np.concatenate([torch.cat([torch.cos(s * base).mean(0), torch.sin(s * base).mean(0)]).double().cpu().numpy()
                           for s in CFG['SIGMAS']])

def pct(a, q): return float(np.percentile(a, q))
SIMPLE = np.array([[c[0], c[1], c[2], pct(p[:, 1], 95), pct(p[:, 2], 95) - pct(p[:, 2], 5),
                    pct(p[:, 0], 95) - pct(p[:, 0], 5), a] for p, c, a in zip(PINNA, CANAL, AREA)])
t0 = time.time()
BLOCKS = {'simple (ear)': SIMPLE,
          'pinna RFF, canal frame': np.stack([rff(p) for p in PINNA]),
          'pinna RFF, ICP-aligned': np.stack([rff(q) for q in ALIGNED]),
          'pinna correspondence': CORR.reshape(len(CORR), -1)}
print(f'特征 {time.time()-t0:.0f} s：' + ', '.join(f'{k} {v.shape[1]} 维' for k, v in BLOCKS.items()))

def zscore(X):
    sd = X.std(0); keep = sd > 1e-12 * max(sd.max(), 1e-300)
    return (X[:, keep] - X[:, keep].mean(0)) / sd[keep]
GB = {k: (lambda Z: Z @ Z.T / Z.shape[1])(zscore(X)) for k, X in BLOCKS.items()}
CONFIGS = {k: [k] for k in BLOCKS}
CONFIGS['simple + pinna RFF aligned'] = ['simple (ear)', 'pinna RFF, ICP-aligned']
CONFIGS['simple + correspondence'] = ['simple (ear)', 'pinna correspondence']
GRAM = {n: sum(GB[b] for b in bl) for n, bl in CONFIGS.items()}

In [ ]:
# ==========================================================================
# Cell 7 — B：形状上的邻居，是不是 z 上的邻居？（不做回归）
# 每侧分别算（左耳和左耳比，右耳和右耳比），两侧结果取平均；统计单位是人
# ==========================================================================
from scipy.stats import spearmanr, rankdata
import matplotlib.pyplot as plt

def dist(X, standardise=True):
    Xz = zscore(X) if standardise else X - X.mean(0); sq = (Xz ** 2).sum(1)
    return np.sqrt(np.maximum(sq[:, None] + sq[None] - 2 * Xz @ Xz.T, 0))
iu = np.triu_indices(NS, 1)
SHAPE_FOR_B = {'pinna correspondence': BLOCKS['pinna correspondence'], 'simple (ear)': SIMPLE}
prng = np.random.default_rng(CFG['SEED'] + 7)
PERMS = [prng.permutation(NS) for _ in range(CFG['N_MANTEL'])]
N_KNN_PERM = min(200, CFG['N_MANTEL'])          # k 近邻的零分布用前 200 个置换
B_RES = {}
for name, X in SHAPE_FOR_B.items():
    rho_s, knn_s, null_rho, null_knn = [], {k: [] for k in CFG['KNN_K']}, np.zeros(CFG['N_MANTEL']), {k: np.zeros(CFG['N_MANTEL']) for k in CFG['KNN_K']}
    for side in (0, 1):
        r = np.flatnonzero(SIDE_OF_ROW == side)
        Ds, Dz = dist(X[r]), dist(Y[r], standardise=False)     # z: raw latent distance
        rz = rankdata(Dz[iu])
        rho_s.append(np.corrcoef(rankdata(Ds[iu]), rz)[0, 1])
        for b, p in enumerate(PERMS):
            null_rho[b] += np.corrcoef(rankdata(Ds[np.ix_(p, p)][iu]), rz)[0, 1] / 2
        np.fill_diagonal(Ds, np.inf); np.fill_diagonal(Dz, np.inf)
        for k in CFG['KNN_K']:
            ns, nz = np.argsort(Ds, 1)[:, :k], np.argsort(Dz, 1)[:, :k]
            knn_s[k].append(np.mean([len(set(a) & set(b)) / k for a, b in zip(ns, nz)]))
            for b, p in enumerate(PERMS[:N_KNN_PERM]):
                nsp = np.argsort(Ds[np.ix_(p, p)], 1)[:, :k]
                null_knn[k][b] += np.mean([len(set(a) & set(c)) / k for a, c in zip(nsp, nz)]) / 2
    rho = float(np.mean(rho_s)); p_rho = (1 + (null_rho >= rho).sum()) / (1 + CFG['N_MANTEL'])
    B_RES[name] = dict(spearman=rho, p=float(p_rho), knn={})
    print(f'{name}: 两两距离的秩相关（形状 vs z）ρ = {rho:+.3f}，置换 p = {p_rho:.3f}')
    for k in CFG['KNN_K']:
        ov = float(np.mean(knn_s[k])); nb = null_knn[k][:N_KNN_PERM]
        pk = (1 + (nb >= ov).sum()) / (1 + N_KNN_PERM)
        B_RES[name]['knn'][k] = dict(overlap=ov, chance=k / (NS - 1), p=float(pk))
        print(f'    形状最近的 {k:2d} 人里也是 z 最近的 {k} 人：{ov:.1%}（随机 {k/(NS-1):.1%}，置换 p = {pk:.3f}）')

# 「一维排序」：形状第一主成分 vs z 第一主成分（左耳）
r = np.flatnonzero(SIDE_OF_ROW == 0)
def pc1(X, standardise=True):
    Xz = zscore(X) if standardise else X - X.mean(0)
    return np.linalg.svd(Xz - Xz.mean(0), full_matrices=False)[0][:, 0]
s1, z1 = pc1(BLOCKS['pinna correspondence'][r]), pc1(Y[r], standardise=False)
rho1 = spearmanr(s1, z1).correlation
plt.figure(figsize=(4.6, 4)); plt.scatter(s1, z1, s=10)
plt.xlabel('pinna shape PC1 (left ears)'); plt.ylabel('latent z PC1 (left ears)'); plt.title(f'Spearman rho = {rho1:+.2f}')
plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'pc1_shape_vs_z.png'), dpi=150); plt.show()
B_RES['pc1_spearman_left'] = float(rho1)

In [ ]:
# ==========================================================================
# Cell 8 — C：回归（FFM_3_2_1 的折、按人分组、核 ridge、折内目标 PCA）
# ==========================================================================
def prep(tr, va):
    my = Y[tr].mean(0); U, s, Vt = np.linalg.svd(Y[tr] - my, full_matrices=False)
    k = int((s > s[0] * 1e-9).sum()); V = Vt[:k].T; Yv = Y[va] - my; Yvp = Yv @ V
    return dict(tr=tr, va=va, my=my, V=V, Ytr=U[:, :k] * s[:k], Yv=Yvp,
                resid=float((Yv ** 2).sum() - (Yvp ** 2).sum()), sst=float((Yv ** 2).sum()))
irng = np.random.default_rng(CFG['SEED'] + 3); PLAN = []
for f in FOLDS:
    va = np.flatnonzero(FOLD[SUBJ_OF_ROW] == f); tr_s = np.unique(SUBJ_OF_ROW[FOLD[SUBJ_OF_ROW] != f])
    inner = [prep(np.flatnonzero(np.isin(SUBJ_OF_ROW, np.setdiff1d(tr_s, v))), np.flatnonzero(np.isin(SUBJ_OF_ROW, v)))
             for v in np.array_split(irng.permutation(tr_s), CFG['N_INNER'])]
    PLAN.append((prep(np.flatnonzero(FOLD[SUBJ_OF_ROW] != f), va), inner))

def path(G, sp, al):
    tr, va = sp['tr'], sp['va']; Gtt, Gvt = G[np.ix_(tr, tr)], G[np.ix_(va, tr)]
    cm, tot = Gtt.mean(0), Gtt.mean(); K = Gtt - cm[None] - cm[:, None] + tot; Kv = Gvt - cm[None] - Gvt.mean(1)[:, None] + tot
    lam, Q = np.linalg.eigh(K); lam = np.clip(lam, 0, None); sc = np.trace(K) / len(tr)
    B = Q.T @ sp['Ytr']; KQ = Kv @ Q
    return [(KQ / (lam + a * sc)) @ B for a in al]
def run(G, want_pred=False):
    A = CFG['ALPHA_REL']; sse = sst = 0.0; pred = np.zeros_like(Y) if want_pred else None; edge = 0
    for outer, inner in PLAN:
        tot = np.sum([[float(((sp['Yv'] - p) ** 2).sum()) for p in path(G, sp, A)] for sp in inner], 0)
        j = int(np.argmin(tot)); edge += j in (0, len(A) - 1)
        p = path(G, outer, [A[j]])[0]; sse += float(((outer['Yv'] - p) ** 2).sum()) + outer['resid']; sst += outer['sst']
        if want_pred: pred[outer['va']] = outer['my'] + p @ outer['V'].T
    return 1 - sse / sst, pred, edge

_Xr = np.random.default_rng(1).standard_normal((2 * NS, 300)); print(f'自检：随机特征 CV-R² {run(_Xr @ _Xr.T / 300)[0]:+.4f}（应 ≤ ~0）')
RES = {}
for n, G in GRAM.items():
    r2, pred, edge = run(G, True); RES[n] = dict(r2=r2, pred=pred)
    print(f'{n:30s} CV-R²(z) {r2:+.4f}' + (f'   ★ {edge} 折 α 在网格边上' if edge else ''))
print(f"{RIDGE_NAME:30s} CV-R²(z) {cv_r2_pred(RIDGE_BEST):+.4f}   ← 要比的对象")
BEST = max(RES, key=lambda k: RES[k]['r2'])
t0 = time.time(); prng = np.random.default_rng(CFG['SEED'] + 1); NULL = []
for b in range(CFG['N_PERM']):
    p = prng.permutation(NS); rr = np.array([2 * p[k] + sd for k in range(NS) for sd in (0, 1)])
    NULL.append(max(run(G[np.ix_(rr, rr)])[0] for G in GRAM.values()))
NULL = np.array(NULL); P_MAX = (1 + (NULL >= RES[BEST]['r2']).sum()) / (1 + len(NULL))
print(f'最好：{BEST} {RES[BEST]["r2"]:+.4f} | 最大值置换零分布 95% 分位 {np.percentile(NULL,95):+.4f}，p = {P_MAX:.3f}  ({time.time()-t0:.0f} s)')

In [ ]:
# ==========================================================================
# Cell 9 — 端到端 dB · LSD；与 FFM_3_2_1 最好 ridge 逐人配对 → 按事先规则下结论
# ==========================================================================
import healpy as hp
ck = torch.load(PATHS['ckpt'], map_location=device, weights_only=False)
ae = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32, num_datasets=4,
                          num_bins=N_BINS, tie_encoder_film=True).to(device)
ae.load_state_dict(ck['model_state_dict']); ae.eval(); del ck
def find_dir(root, target):
    c = []
    for dp, _, fns in os.walk(root):
        if '__MACOSX' in dp: continue
        if os.path.basename(dp) == target: c.append((sum(f.endswith('.npy') for f in fns), dp))
    c.sort(reverse=True); return c[0][1] if c else None
if PATHS['sonicom'].endswith('.zip'):
    SON_OUT = '/content/dataset/sonicom'; os.makedirs(SON_OUT, exist_ok=True)
    if find_dir(SON_OUT, 'npy_left') is None:
        with zipfile.ZipFile(PATHS['sonicom']) as z: z.extractall(SON_OUT)
else:
    SON_OUT = PATHS['sonicom']
DIRS = (find_dir(SON_OUT, 'npy_left'), find_dir(SON_OUT, 'npy_right'))
_th, _ph = hp.pix2ang(16, np.arange(3072)); MIRROR = hp.ang2pix(16, _th, (-_ph) % (2 * np.pi))
MEAS = (90 - np.degrees(_th)) >= -45
FREQS = np.arange(N_BINS) * 44100 / 256; NOTCH = (FREQS >= CFG['NOTCH_HZ'][0]) & (FREQS <= CFG['NOTCH_HZ'][1])
X = np.zeros((2 * NS, N_BINS, 3072), np.float32)
for k, s in enumerate(DEV):
    for side in (0, 1):
        x = np.load(os.path.join(DIRS[side], f"{s}_{('left', 'right')[side]}_full.npy")).astype(np.float32)[:N_BINS]
        X[2 * k + side] = x[:, MIRROR] if side == 1 else x

@torch.no_grad()
def decode_rows(Zr):
    out = np.zeros((len(Zr), N_BINS, 3072), np.float32)
    for b in range(0, len(Zr), 16):
        z = torch.from_numpy(Zr[b:b + 16].reshape(-1, N_BINS, 64).astype(np.float32)).to(device)
        out[b:b + 16] = ae.decoder(z, torch.full((len(z),), CFG['DID'], device=device)).cpu().numpy()
    return out
def per_ear(P):
    e2 = (P - X) ** 2; d = np.sqrt(e2.mean(1))
    return dict(mse=e2.mean((1, 2)), mse_n=e2[:, NOTCH].mean((1, 2)), lsd=d.mean(1))
subj = lambda v: v.reshape(NS, 2).mean(1)
ROWS = {}
for name, pred in ((RIDGE_NAME, RIDGE_BEST), (f'pinna: {BEST}', RES[BEST]['pred'])):
    pe = per_ear(decode_rows(pred))
    ROWS[name] = dict(pe=pe, db=float(np.sqrt(subj(pe['mse']).mean())), db_notch=float(np.sqrt(subj(pe['mse_n']).mean())),
                      lsd=float(pe['lsd'].mean()))
for n, r in ROWS.items():
    print(f"{n:52s} pooled {r['db']:.3f} dB | 6–12 kHz {r['db_notch']:.3f} | LSD {r['lsd']:.3f}")
a, b = subj(ROWS[f'pinna: {BEST}']['pe']['mse']), subj(ROWS[RIDGE_NAME]['pe']['mse'])
BOOT = np.random.default_rng(CFG['SEED']).integers(0, NS, (CFG['N_BOOT'], NS))
d = np.sqrt(a[BOOT].mean(1)) - np.sqrt(b[BOOT].mean(1)); ci = np.percentile(d, [2.5, 97.5])
diff = float(np.sqrt(a.mean()) - np.sqrt(b.mean()))
print(f'\n耳廓版 − FFM_3_2_1 最好 ridge：{diff:+.3f} dB，95% 区间 [{ci[0]:+.3f}, {ci[1]:+.3f}]（负数 = 耳廓版更好）')
CLOSE = ci[1] >= 0
print('事先规则：' + ('★ 区间包含 0 或在劣侧 —— mesh2z 正式结束' if CLOSE else
                      '区间整体在 0 以下 —— 耳廓版确有改善；按规则 mesh2z 仍然收尾，但这条改善写进 thesis'))

In [ ]:
# ==========================================================================
# Cell 10 — 导出
# ==========================================================================
out = dict(cfg=CFG, n_subjects=NS, pinna_points_median=float(np.median(NPTS)),
           similarity=B_RES, cv_r2={k: v['r2'] for k, v in RES.items()}, ridge_ref=float(cv_r2_pred(RIDGE_BEST)),
           best=BEST, p_max=float(P_MAX), null_p95=float(np.percentile(NULL, 95)),
           e2e={k: {kk: vv for kk, vv in v.items() if kk != 'pe'} for k, v in ROWS.items()},
           paired_db=dict(diff=diff, ci=[float(c) for c in ci]), close_mesh2z=bool(CLOSE))
with open(os.path.join(OUT_DIR, 'results.json'), 'w') as f:
    json.dump(out, f, indent=1, ensure_ascii=False, default=float)
print(f'-> {OUT_DIR}/results.json, pc1_shape_vs_z.png')

## 读结果之前
1. Cell 3 对齐检查、Cell 8 随机特征自检必须通过。
2. Cell 5：ICP 的旋转应是几度量级的微调；几十度说明模板或初始化有问题。
3. Cell 7 是不靠模型的检验：若形状邻居和 z 邻居的重合只是略高于随机，任何回归都不会好太多。
4. Cell 9 的结论按事先规则读，不因结果改规则。